# Notebook 02 — JSON Customer Profiles Ingestion & EDA

**Source:** `customer_profiles.json` (375,837 documents)  
**Owner:** Jad Badran + Elie Estephan (original) | Consolidated by Elie Khairallah  
**Rubric:** §4.1, §4.3, §4.4

In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, os.path.abspath('..'))
os.chdir(os.path.abspath('..'))
Path('data/audit').mkdir(exist_ok=True)
print('Working directory:', os.getcwd())

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Libraries loaded')

## 1. Ingestion

The JSON file is a wrapper object with records at `obj['data']` (not NDJSON). The ingestion module validates the record count against the metadata header.

In [ ]:
from src.ingestion import ingest_json
meta = ingest_json.get_metadata()
print('File metadata:')
for k, v in meta.items():
    if k not in ('fields', 'account_status_distribution'):
        print(f'  {k}: {v}')

In [ ]:
df_raw = ingest_json.load_raw()
print(f'Shape: {df_raw.shape}')
print(df_raw.dtypes)

In [ ]:
print('Null counts:', df_raw.isnull().sum().sum())
print('Duplicate wallet_ids:', df_raw['wallet_id'].duplicated().sum())
print('WLT format check:', df_raw['wallet_id'].str.match(r'^WLT-\d{8}$').all())

## 2. Cleaning

In [ ]:
from src.cleaning import clean_json
df_clean, report = clean_json.clean(df_raw)
print('Actions:')
for a in report['actions']:
    print(f'  • {a}')

## 3. EDA

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
df_clean['account_status'].value_counts().plot(kind='bar', ax=axes[0], color=['#55A868','#C44E52','#DD8452'], edgecolor='black')
axes[0].set_title('Account Status', fontweight='bold'); axes[0].tick_params(axis='x', rotation=0)
df_clean['support_tier'].value_counts().plot(kind='bar', ax=axes[1], color=['#4C72B0','#DD8452','#55A868'], edgecolor='black')
axes[1].set_title('Support Tier', fontweight='bold'); axes[1].tick_params(axis='x', rotation=0)
df_clean['gender'].value_counts().plot(kind='pie', ax=axes[2], autopct='%1.1f%%', colors=['#4C72B0','#DD8452'])
axes[2].set_title('Gender Split', fontweight='bold'); axes[2].set_ylabel('')
plt.tight_layout()
fig.savefig('data/audit/fig_04_json_demographics.png', dpi=100, bbox_inches='tight')
plt.show()

**Interpretation:** 65% of accounts are active, 27% dormant — dormant accounts are key churn candidates. Silver is the dominant support tier. Gender split is near-equal (50.1% Female), confirming the synthetic data generator's design.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df_clean['age'].plot(kind='hist', bins=30, ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Age Distribution', fontweight='bold'); axes[0].set_xlabel('Age')
lang = df_clean['preferred_language'].value_counts()
lang.plot(kind='barh', ax=axes[1], color='mediumpurple', edgecolor='black')
axes[1].set_title('Preferred Language', fontweight='bold')
plt.tight_layout()
fig.savefig('data/audit/fig_05_json_age_lang.png', dpi=100, bbox_inches='tight')
plt.show()

**Interpretation:** Age distribution is roughly uniform between 18–65 (synthetic design). English dominates (35%), followed by Hausa (25%) and Yoruba (20%) — reflecting Nigeria's linguistic diversity.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
top_states = df_clean['state'].value_counts().head(10)
top_states.plot(kind='barh', ax=ax, color='teal', edgecolor='black')
ax.set_title('Top 10 States by Profile Count', fontweight='bold')
ax.set_xlabel('Profile Count')
plt.tight_layout()
fig.savefig('data/audit/fig_06_json_states.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. Save Processed Output

In [ ]:
from pathlib import Path
Path('data/processed').mkdir(exist_ok=True)
Path('data/samples').mkdir(exist_ok=True)

df_clean.to_json('data/processed/json_cleaned.json', orient='records', lines=True, force_ascii=False)
print('Saved: data/processed/json_cleaned.json')

df_clean.sample(100, random_state=42).to_json('data/samples/json_sample_100.json', orient='records', indent=2, force_ascii=False)
print('Saved: data/samples/json_sample_100.json')